# Fruit Juice Storage: Data Analysis with Python

This notebook is a step-by-step classroom demonstration for beginners.

**Audience:** Students who know VS Code and basic Python syntax.

**Learning goals:**

- understand rows, columns and variables in a CSV table;
- read a CSV file with pandas;
- inspect data with `head()`, `shape`, `columns`, `info()` and `isna()`;
- use `dropna()`, `groupby()` and `mean()` for a clear analysis purpose;
- create and interpret line, grouped-bar and scatter plots.

**Important:** These are experimental data (synthesized data) for teaching. They do not represent real
products, laboratory measurements, clinical evidence, food-safety thresholds,
regulatory advice or causal scientific conclusions.

## The experiment

The dataset contains:

`3 juice types × 3 storage temperatures × 4 storage days × 3 batches = 108 rows`

Each row represents one experimental sample. We will ask two simple questions:

1. How does storage temperature affect vitamin C in these experimental data?
2. How does microbial load change with storage time and temperature?

The workflow is:

`read → inspect → check missing values → clean for a purpose → summarize → visualize → interpret carefully`

## 0. Install necessary libraries
Run the cell below first to check whether the required libraries are installed and to display their versions.  
If you get an import error, run the installation cell below and then rerun the version check.

In [1]:
import numpy
import matplotlib
import pandas
print(numpy.__version__)
print(pandas.__version__)
print(matplotlib.__version__)

2.5.1
3.0.5
3.11.1


In [ ]:
# !pip uninstall numpy

In [4]:
!pip install numpy
!pip install pandas
!pip install matplotlib

## 1. Import the libraries

Run this cell first. `pandas` works with tables and `matplotlib` creates plots.
The short names `pd` and `plt` are conventions used throughout this notebook.

In [5]:
%matplotlib inline

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

## 2. Set the paths

Run this notebook with the project root as the current working directory.
The notebook reads the existing CSV and saves notebook-specific plots in
the `outputs` folder.

In [6]:
project_root = Path.cwd()
data_path = project_root / "data" / "fruit_juice_storage.csv"
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Data file: {data_path}")
print(f"Output folder: {output_dir}")

Data file: /Users/zhenjiao-ucd/Desktop/UCD-Teaching-learning-training-etc/AI-in-science-and-engineering/Demo/python-data-analysis/data/fruit_juice_storage.csv
Output folder: /Users/zhenjiao-ucd/Desktop/UCD-Teaching-learning-training-etc/AI-in-science-and-engineering/Demo/python-data-analysis/outputs


## 3. Read the CSV file

`pd.read_csv()` reads comma-separated text and creates a pandas DataFrame.
A DataFrame is a table with labelled rows and columns.

In [7]:
df = pd.read_csv(data_path)
df.head()

,sample_id,juice_type,storage_temp_C,storage_day,batch,vitamin_c_mg_per_100ml,pH,microbial_load_log10,turbidity_NTU
0,S001,orange,4,0,1,52.11,3.62,1.90,13.2
1,S002,orange,4,0,2,51.32,3.62,1.86,12.1
2,S003,orange,4,0,3,51.99,3.63,1.91,13.1
3,S004,orange,4,2,1,50.62,3.67,2.06,12.7
4,S005,orange,4,2,2,50.73,3.61,2.09,13.4


In [8]:
df.head(10)

,sample_id,juice_type,storage_temp_C,storage_day,batch,vitamin_c_mg_per_100ml,pH,microbial_load_log10,turbidity_NTU
0,S001,orange,4,0,1,52.11,3.62,1.90,13.2
1,S002,orange,4,0,2,51.32,3.62,1.86,12.1
2,S003,orange,4,0,3,51.99,3.63,1.91,13.1
3,S004,orange,4,2,1,50.62,3.67,2.06,12.7
4,S005,orange,4,2,2,50.73,3.61,2.09,13.4
5,S006,orange,4,2,3,50.54,3.62,2.12,13.4
6,S007,orange,4,4,1,49.05,3.62,2.25,14.4
7,S008,orange,4,4,2,49.34,3.64,2.36,14.3
8,S009,orange,4,4,3,49.02,3.61,2.25,14.9
9,S010,orange,4,6,1,47.76,3.59,2.33,15.0


## 4. Inspect the size and column names

`shape` returns `(number_of_rows, number_of_columns)`. The column names tell
us what each variable means.

In [9]:
df.shape

(108, 9)

In [10]:
df.columns

Index(['sample_id', 'juice_type', 'storage_temp_C', 'storage_day', 'batch',
       'vitamin_c_mg_per_100ml', 'pH', 'microbial_load_log10',
       'turbidity_NTU'],
      dtype='str')

## 5. Inspect data types and missing values

`info()` shows column names, data types and how many values are present.
The dataset deliberately contains two missing `pH` values and two missing
`turbidity_NTU` values.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   sample_id               108 non-null    str    
 1   juice_type              108 non-null    str    
 2   storage_temp_C          108 non-null    int64  
 3   storage_day             108 non-null    int64  
 4   batch                   108 non-null    int64  
 5   vitamin_c_mg_per_100ml  108 non-null    float64
 6   pH                      106 non-null    float64
 7   microbial_load_log10    108 non-null    float64
 8   turbidity_NTU           106 non-null    float64
dtypes: float64(4), int64(3), str(2)
memory usage: 7.7 KB


In [ ]:
df.isna() # binary mask of missing values

In [ ]:
df.isna().sum() # summary of missing values for each column

sample_id                 0
juice_type                0
storage_temp_C            0
storage_day               0
batch                     0
vitamin_c_mg_per_100ml    0
pH                        2
microbial_load_log10      0
turbidity_NTU             2
dtype: int64

**Important distinction:** a missing value is not the same as zero. In pandas,
missing numeric values are usually displayed as `NaN`.

In [ ]:
df[df["pH"].isna()] # show the rows with missing pH values

,sample_id,juice_type,storage_temp_C,storage_day,batch,vitamin_c_mg_per_100ml,pH,microbial_load_log10,turbidity_NTU
10,S011,orange,4,6,2,48.06,NaN,2.34,14.7
70,S071,apple,35,6,2,35.32,NaN,5.70,31.5


In [ ]:
df[df["turbidity_NTU"].isna()] # show the rows with missing turbidity_NTU values

In [ ]:
df[["pH", "turbidity_NTU"]].describe() # show summary statistics for pH and turbidity_NTU columns


,pH,turbidity_NTU
count,106.000000,106.000000
mean,3.548302,19.129245
std,0.206640,5.963156
min,3.140000,11.700000
25%,3.330000,14.700000
50%,3.605000,17.750000
75%,3.747500,22.550000
max,3.860000,42.400000


## 6. Filter a scientific subset

Analysis starts with a question. Here we select only orange juice so that
we can examine vitamin C over storage time at three temperatures.

In [13]:
orange_df = df[df["juice_type"] == "orange"]
orange_df.head()

,sample_id,juice_type,storage_temp_C,storage_day,batch,vitamin_c_mg_per_100ml,pH,microbial_load_log10,turbidity_NTU
0,S001,orange,4,0,1,52.11,3.62,1.90,13.2
1,S002,orange,4,0,2,51.32,3.62,1.86,12.1
2,S003,orange,4,0,3,51.99,3.63,1.91,13.1
3,S004,orange,4,2,1,50.62,3.67,2.06,12.7
4,S005,orange,4,2,2,50.73,3.61,2.09,13.4


## 7. Clean data for a stated purpose

We make a copy before cleaning. The following complete-case example removes
rows missing either `pH` or `turbidity_NTU`. It is useful for demonstrating
`copy()` and `dropna()`, but it is not used for the vitamin C summary.

In [ ]:
clean_df = df.copy()
clean_df = clean_df.dropna(subset=["pH", "turbidity_NTU"])

clean_df.shape

The plotting steps below use the original `df` and remove only the columns
required by each particular plot. This avoids dropping a vitamin C value just
because an unrelated `pH` value is missing.

## 8. Calculate a grouped summary

We want one average vitamin C value for each juice type and storage
temperature. `groupby()` creates the groups, and `mean()` calculates the
average within each group.

In [ ]:
summary = (
    df.groupby(["juice_type", "storage_temp_C"])["vitamin_c_mg_per_100ml"]
    .mean()
    .reset_index()
    .round(2)
)

summary

The summary has 9 rows: 3 juice types × 3 temperatures. It is a compact
description of the data, not a replacement for looking at individual samples.

## 9. Plot vitamin C over time

This line chart uses orange juice and shows one line for each storage
temperature. Each point is the mean vitamin C at that storage day.

In [ ]:
orange_plot_df = orange_df[
    ["storage_temp_C", "storage_day", "vitamin_c_mg_per_100ml"]
].dropna()

plt.figure(figsize=(8, 5))

for temperature in sorted(orange_plot_df["storage_temp_C"].unique()):
    temperature_df = orange_plot_df[
        orange_plot_df["storage_temp_C"] == temperature
    ]
    vitamin_c_by_day = (
        temperature_df.groupby("storage_day")["vitamin_c_mg_per_100ml"]
        .mean()
    )
    plt.plot(
        vitamin_c_by_day.index,
        vitamin_c_by_day.values,
        marker="o",
        label=f"{temperature} °C",
    )

plt.title("Vitamin C in orange juice during storage")
plt.xlabel("Storage day")
plt.ylabel("Vitamin C (mg per 100 mL)")
plt.legend(title="Temperature")
plt.grid(alpha=0.3)
plt.tight_layout()

vitamin_c_plot_path = output_dir / "vitamin_c_over_time_notebook.png"
plt.savefig(vitamin_c_plot_path, dpi=150)
plt.show()
plt.close()

print(f"Saved: {vitamin_c_plot_path}")

In these experimental data, vitamin C generally decreases over storage time,
and warmer storage conditions tend to produce lower values at later days.

## 10. Plot microbial load by storage day and temperature

The grouped bar chart makes it easy to compare temperatures at the same day.
The vertical axis is an experimental `log10 CFU/mL` representation, not a real
food-safety threshold.

In [ ]:
microbial_plot_df = df[
    ["storage_temp_C", "storage_day", "microbial_load_log10"]
].dropna()

microbial_means = (
    microbial_plot_df
    .groupby(["storage_day", "storage_temp_C"])["microbial_load_log10"]
    .mean()
    .unstack("storage_temp_C")
)

ax = microbial_means.plot(kind="bar", figsize=(9, 5))
ax.set_title("Average microbial load by storage day and temperature")
ax.set_xlabel("Storage day")
ax.set_ylabel("Microbial load (log10 scale)")
ax.legend(title="Temperature")
plt.tight_layout()

microbial_plot_path = output_dir / "microbial_load_by_temperature_notebook.png"
plt.savefig(microbial_plot_path, dpi=150)
plt.show()
plt.close()

print(f"Saved: {microbial_plot_path}")

In these experimental data, microbial load generally increases with storage
time. The warmer conditions tend to be higher at later storage days.

## 11. Plot vitamin C versus turbidity

A scatter plot contains one point for each complete vitamin-C/turbidity pair.
It can show a visual association in these experimental observations, but it cannot prove
that one variable causes the other.

In [ ]:
scatter_df = df[["vitamin_c_mg_per_100ml", "turbidity_NTU"]].dropna()
scatter_df.shape

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    scatter_df["vitamin_c_mg_per_100ml"],
    scatter_df["turbidity_NTU"],
    alpha=0.7,
    edgecolor="white",
    label="Experimental measurements",
)

plt.title("Vitamin C and turbidity")
plt.xlabel("Vitamin C (mg per 100 mL)")
plt.ylabel("Turbidity (NTU)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

scatter_plot_path = output_dir / "vitamin_c_vs_turbidity_notebook.png"
plt.savefig(scatter_plot_path, dpi=150)
plt.show()
plt.close()

print(f"Saved: {scatter_plot_path}")

## Optional extension: inspect the teaching outlier

If there is time, locate the documented high microbial-load teaching value.
It is an invented value for demonstrating inspection, not a real safety signal.

In [ ]:
df[df["sample_id"] == "S108"]

## Interpretation exercise

Write two observations in your own words:

1. What pattern do you see in vitamin C as storage time and temperature change?
2. What pattern do you see in microbial load as storage time and temperature change?

Start your answer with:

> In these experimental data, ...

This wording reminds us that the results are patterns in teaching data based on
synthesized measurements, not validated health, food-safety or clinical evidence.

In [ ]:
print("Rows and columns:", df.shape)
print("\nVitamin C summary:")
print(summary.to_string(index=False))

## Recap

The reusable data-analysis workflow is:

`import → read → inspect → check missingness → clean for a purpose → group and average → plot → communicate limitations`

The main lesson is not to memorize commands. Start with a question, make
each operation visible, and explain what the data can and cannot support.